In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_repo_daily_activity"

ECOSYSTEM_DAILY_TABLE = (
    f"{CATALOG}.{SCHEMA}.gold_ecosystem_daily_summary"
)

In [0]:
ecosystem_daily_summary_df = (
    spark.table(GOLD_TABLE)
    .groupBy("event_date")
    .agg(
        F.sum("total_events").alias("total_events"),

        F.countDistinct("repo_id").alias("active_repositories"),

        F.sum("bot_events").alias("bot_events"),

        F.sum("human_or_unknown_events")
            .alias("human_or_unknown_events"),

        F.sum("newly_observed_contributors")
            .alias("newly_observed_contributors"),

        F.sum("returning_contributors")
            .alias("returning_contributors"),

        F.expr(
            "percentile_approx(top_contributor_share, 0.5)"
        ).alias("median_top_contributor_share"),

        F.sum("stars").alias("stars"),

        F.sum("forks").alias("forks"),

        F.sum("prs_opened").alias("prs_opened"),

        F.sum("issues_opened").alias("issues_opened")
    )
    .orderBy("event_date")
)

In [0]:
ecosystem_daily_summary_df.show(
    7,
    truncate=False
)

+----------+------------+-------------------+----------+-----------------------+---------------------------+----------------------+----------------------------+------+-----+----------+-------------+
|event_date|total_events|active_repositories|bot_events|human_or_unknown_events|newly_observed_contributors|returning_contributors|median_top_contributor_share|stars |forks|prs_opened|issues_opened|
+----------+------------+-------------------+----------+-----------------------+---------------------------+----------------------+----------------------------+------+-----+----------+-------------+
|2026-09-01|637429      |86916              |31320     |606109                 |68520                      |0                     |1.0                         |2563  |597  |5391      |1581         |
|2026-09-02|45967       |30950              |18158     |27809                  |19472                      |2907                  |1.0                         |2984  |702  |6709      |2127         |
|2026

In [0]:
ecosystem_daily_consumption_df = (
    ecosystem_daily_summary_df
    .withColumn(
        "bot_event_share",
        F.col("bot_events") / F.col("total_events")
    )
    .withColumn(
        "returning_contributor_share",
        F.when(
            (
                F.col("newly_observed_contributors") +
                F.col("returning_contributors")
            ) > 0,
            F.col("returning_contributors") /
            (
                F.col("newly_observed_contributors") +
                F.col("returning_contributors")
            )
        )
    )
)

In [0]:
ecosystem_daily_consumption_df.select(
    "event_date",
    "total_events",
    "active_repositories",
    "newly_observed_contributors",
    "returning_contributors",
    F.round(
        F.col("returning_contributor_share") * 100,
        2
    ).alias("returning_share_pct"),
    F.round(
        F.col("bot_event_share") * 100,
        2
    ).alias("bot_event_share_pct")
).show(
    7,
    truncate=False
)

+----------+------------+-------------------+---------------------------+----------------------+-------------------+-------------------+
|event_date|total_events|active_repositories|newly_observed_contributors|returning_contributors|returning_share_pct|bot_event_share_pct|
+----------+------------+-------------------+---------------------------+----------------------+-------------------+-------------------+
|2026-09-01|637429      |86916              |68520                      |0                     |0.0                |4.91               |
|2026-09-02|45967       |30950              |19472                      |2907                  |12.99              |39.5               |
|2026-09-03|159255      |66043              |45642                      |7017                  |13.33              |40.94              |
|2026-09-04|1499201     |286590             |298088                     |31099                 |9.45               |38.57              |
|2026-09-05|1867891     |337445          

In [0]:
(
    ecosystem_daily_consumption_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ECOSYSTEM_DAILY_TABLE)
)

In [0]:
consumption_df = spark.table(
    ECOSYSTEM_DAILY_TABLE
)

print(
    "Consumption rows:",
    consumption_df.count()
)

print(
    "Distinct dates:",
    consumption_df
    .select("event_date")
    .distinct()
    .count()
)

Consumption rows: 7
Distinct dates: 7


In [0]:
# Daily ecosystem activity
display(
    consumption_df
    .select(
        "event_date",
        "total_events"
    )
    .orderBy("event_date")
)

event_date,total_events
2026-09-01,637429
2026-09-02,45967
2026-09-03,159255
2026-09-04,1499201
2026-09-05,1867891
2026-09-06,2504097
2026-09-07,2124296


Databricks visualization. Run in Databricks to view.

In [0]:
# Newly observed vs returning contributors
display(
    consumption_df
    .select(
        "event_date",
        "newly_observed_contributors",
        "returning_contributors"
    )
    .orderBy("event_date")
)

event_date,newly_observed_contributors,returning_contributors
2026-09-01,68520,0
2026-09-02,19472,2907
2026-09-03,45642,7017
2026-09-04,298088,31099
2026-09-05,315140,65321
2026-09-06,406923,96831
2026-09-07,309824,107449


Databricks visualization. Run in Databricks to view.

In [0]:
# Bot activity share
display(
    consumption_df
    .select(
        "event_date",
        "bot_event_share"
    )
    .orderBy("event_date")
)

event_date,bot_event_share
2026-09-01,0.049134884041987424
2026-09-02,0.3950225161528923
2026-09-03,0.4093560641738093
2026-09-04,0.3857468077996213
2026-09-05,0.34677558808303055
2026-09-06,0.276795587391383
2026-09-07,0.15755148999951044


Databricks visualization. Run in Databricks to view.